In [1]:
# IMPORT, CONFIG

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

from scrapers import MatchScraper, H2HScraper
from scrapers.base_scraper import BaseScraper
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

rankings = pd.read_csv("data/all_rankings2025.csv")
rankings.date = pd.to_datetime(rankings.date)

tours = {'8292': {'name': 'ESL Pro League Season 21',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-esl-pro-league-season-21/results/'},
         '7907': {'name': 'BLAST Open London 2025',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-blast-open/results/'},
         '8038': {'name': 'IEM Cologne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-intel-extreme-masters-cologne/results/'},
         '8036': {'name': 'IEM Melbourne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-iem-melbourne/results/'}
        }


In [2]:
# 3. TARGET MATCHES SCRAPING

def _target_match_scrape(event_id):

    print("\n" + "="*60)
    print("1️⃣ TARGET MATCHES SCRAPING")
    print("="*60)

    try:
        scraper = MatchScraper(headless=False)
        target_matches = scraper.scrape_event_matches(event_id)
        
        if target_matches.empty:
            raise ValueError("❌ Nincs target match!")
        
        # ordinal ragok eltávolítása
        target_matches["date_clean"] = target_matches["date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))

        # dátummá alakítás
        target_matches["date_parsed"] = pd.to_datetime(target_matches["date_clean"], format="%B %d %Y")

        
        print(f"✅ {len(target_matches)} meccs találva")
        display(target_matches.head(3))
        
    except Exception as e:
        print(f"❌ Hiba a target matches scraping közben: {e}")
        
    return target_matches

In [3]:
# 5. MATCH H2H SCRAPING
def _h2h_match_scrape(selected_match):
    print("\n" + "="*60)
    print("3️⃣ MATCH H2H SCRAPING")
    print("="*60)

    try:
        match_url = selected_match['link']
        print(f"🔍 Scraping: {match_url}")
        
        scraper = H2HScraper(headless=False)
        h2h_df = scraper.scrape_match_h2h(match_url)
        
        if h2h_df.empty:
            raise ValueError("❌ H2H scraping sikertelen!")
        
        match_h2h = h2h_df.iloc[0]
        
        print(f"✅ H2H data:")
        display(match_h2h)

        return match_h2h
        
    except Exception as e:
        print(f"❌ Hiba a H2H scraping közben: {e}")

        return pd.DataFrame()

In [4]:
# 6. TEAMS EXTRACTION

def _extract_teams(match_h2h):
    print("\n" + "="*60)
    print("4️⃣ TEAMS EXTRACTION")
    print("="*60)

    teams = {
        'home': {
            'team_id': str(match_h2h.get('home_team_id', 'unknown')),
            'team_name': match_h2h['home_team']
        },
        'away': {
            'team_id': str(match_h2h.get('away_team_id', 'unknown')), 
            'team_name': match_h2h['away_team']
        }
    }

    print(f"✅ Teams extracted:")
    print(f"  Home: {teams['home']['team_name']} (ID: {teams['home']['team_id']})")
    print(f"  Away: {teams['away']['team_name']} (ID: {teams['away']['team_id']})")

    return teams

In [5]:
# 7. TEAM HISTORY SCRAPING HELPER

print("\n" + "="*60)
print("5️⃣ TEAM HISTORY SCRAPING HELPER")
print("="*60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from scrapers.base_scraper import BaseScraper

class TeamHistoryScraper(BaseScraper):
    def scrape_team_matches(self, team_id: str, max_matches: int = 20):
        """Team match history scraping"""
        url = f"https://www.hltv.org/results?team={team_id}"
        self._init_driver()
        self.driver.get(url)

        wait = WebDriverWait(self.driver, 20)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

        self._random_delay()

        matches = []
        all_sublists = self.driver.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Összesen {len(all_sublists)} results-sublist betöltve")

        for sublist in all_sublists:
            if len(matches) >= max_matches:
                break

            try:
                headline = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
                match_date = headline.replace("Results for", "").strip()
            except:
                match_date = None

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                if len(matches) >= max_matches:
                    break

                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    match_id = match_url.split('/')[4]

                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    team1_html = tds[0].get_attribute("innerHTML")
                    won = "team-won" in team1_html

                    try:
                        map_text = a_tag.find_element(By.CSS_SELECTOR, ".map-text").text.strip()
                    except:
                        map_text = "bo1"

                    opponent_name = team2_name if team1_name else team1_name

                    matches.append({
                        "team_id": team_id,
                        "match_id": match_id,
                        "match_date": match_date,
                        "opponent_name": opponent_name,
                        "result": "win" if won else "loss",
                        "score_for": score1,
                        "score_against": score2,
                        "map_type": map_text,
                        "link": match_url
                    })

                except Exception as e:
                    print(f"⚠️ Hiba egy meccs feldolgozásánál: {e}")
                    continue

        self.close()
        return pd.DataFrame(matches)

print("✅ TeamHistoryScraper helper kész")


# 8. TEAM HISTORIES SCRAPING

def _team_history_scrape(teams, selected_match):
    print("\n" + "="*60)
    print("6️⃣ TEAM HISTORIES SCRAPING")
    print("="*60)

    team_histories = {}

    for side in ['home', 'away']:
        team_id = teams[side]['team_id']
        team_name = teams[side]['team_name']
        
        print(f"\n📈 Scraping history: {team_name} (ID: {team_id})")
        
        try:
            scraper = TeamHistoryScraper(headless=False)
            history_df = scraper.scrape_team_matches(team_id, max_matches=100)
            
            if history_df.empty:
                print(f"⚠️ Nincs history adat: {team_name}")
                team_histories[side] = pd.DataFrame()
                continue

            # ordinal ragok eltávolítása
            history_df["date_clean"] = history_df["match_date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))
            # dátummá alakítás
            history_df["date_parsed"] = pd.to_datetime(history_df["date_clean"], format="%B %d %Y")

            # csak a meccs dátuma előttiek (megelőző meccsek)
            history_df = history_df[history_df['date_parsed'] < selected_match['date_parsed']]       

            if len(history_df) < 3:
                print(f"⚠️ Nincs elég history adat: {team_name}")
                team_histories[side] = pd.DataFrame()
                continue

            team_histories[side] = history_df
            
            print(f"✅ {len(history_df)} meccs találva")

            print(f"\n📋 Megelőző 3 meccs:")
            display(history_df.head(3))
            
        except Exception as e:
            print(f"❌ Hiba a history scraping közben: {e}")
            team_histories[side] = pd.DataFrame()
        
    return team_histories


5️⃣ TEAM HISTORY SCRAPING HELPER
✅ TeamHistoryScraper helper kész


In [6]:
# 9. HISTORICAL H2H SCRAPING

# Get rank data function
def get_rank_data(date, team_name):
    team_rankings = rankings[(rankings['date'] < date) & 
                            (rankings['team_name'] == team_name)].reset_index(drop=True)

    # Current (latest) rank & points
    rank = team_rankings.loc[0, 'rank']
    points = team_rankings.loc[0, 'points']
    # Latest rank & points change
    rank_change = team_rankings.loc[0, 'rank'] - team_rankings.loc[1, 'rank']
    points_change = team_rankings.loc[0, 'points'] - team_rankings.loc[1, 'points']

    return({'rank': rank,
            'rank_change': rank_change, 
            'points': points,
            'points_change': points_change}
            )
print("Rank data function initialized.")


def _historical_h2h_scrape(teams, team_histories, selected_match):
    print("\n" + "="*60)
    print("7️⃣ HISTORICAL H2H SCRAPING")
    print("="*60)

    historical_h2h = {}
    N_MATCHES = 3

    rank_data = {}
    for side in ['home', 'away']:
        team_name = teams[side]['team_name']
        team_id = teams[side]['team_id']
        date = selected_match['date_parsed']

        try:
            rank_data[side] = get_rank_data(date, team_name)
        except Exception as e:
            print(f"⚠️ Rank data hiba {team_name} esetén: {e}")
            rank_data[side] = {'rank': None, 'rank_change': None, 'points': None, 'points_change': None}
        
        print(f"\n🔍 {team_name} last {N_MATCHES} matches H2H scraping:")
        
        history = team_histories.get(side, pd.DataFrame())
        
        if history.empty:
            print(f"  ⚠️ Nincs history, skip")
            historical_h2h[side] = {}
            continue
        
        history = history[history['date_parsed'] < date].reset_index(drop=True)
        
        # Last N match URL-ek
        if len(history) < N_MATCHES:
            print(f"  ⚠️ Kevesebb mint {N_MATCHES} meccs a historyban, csak {len(history)} elérhető")
            # Itt is adj hozzá üres dict-et
            historical_h2h[side] = {}
            continue
        
        last_n_matches = history.head(N_MATCHES)
        
        h2h_total = []
        h2h_wins = []
        ratings = []
        rating_stds = []
        adrs = []
        adr_stds = []
        swings = []
        swing_stds = []
        maps_total = []
        maps_wins = []
        maps_picked = []
        maps_avg_score_diffs = []
        ranks_opp = []
        ranks_change = []
        rank_diffs = []
        points_opp = []
        points_change = []
        point_diffs = []
        
        for idx, match in last_n_matches.iterrows():
            match_url = match['link']
            print(f"  [{idx+1}/{N_MATCHES}] Scraping: {match['date_parsed'].date()} vs {match['opponent_name']}\n{match_url}")
            rank_data_side = get_rank_data(match['date_parsed'], team_name)

            try:
                scraper = H2HScraper(headless=False)
                hist_h2h_df = scraper.scrape_match_h2h(match_url)

                if hist_h2h_df.empty:
                    raise ValueError("❌ H2H scraping sikertelen!")
                
                if str(hist_h2h_df['home_team_id'][0]) == str(team_id):
                    h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                    h2h_wins.append(hist_h2h_df['wins_home'][0])
                    ratings.append(hist_h2h_df['home_team_avg_rating'][0])
                    rating_stds.append(hist_h2h_df['home_team_std_rating'][0])
                    adrs.append(hist_h2h_df['home_team_avg_ADR'][0])
                    adr_stds.append(hist_h2h_df['home_team_std_ADR'][0])
                    swings.append(hist_h2h_df['home_team_avg_Swing'][0])
                    swing_stds.append(hist_h2h_df['home_team_std_Swing'][0])
                    maps_total.append(hist_h2h_df['maps_played'][0])
                    maps_wins.append(hist_h2h_df['home_maps_won'][0])
                    maps_picked.append(hist_h2h_df['home_maps_picked'][0])
                    maps_avg_score_diffs.append(hist_h2h_df['map_avg_score_diff'][0])
                    opp = hist_h2h_df['away_team'][0]

                elif str(hist_h2h_df['away_team_id'][0]) == str(team_id):
                    h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                    h2h_wins.append(hist_h2h_df['wins_away'][0])
                    ratings.append(hist_h2h_df['away_team_avg_rating'][0])
                    rating_stds.append(hist_h2h_df['away_team_std_rating'][0])
                    adrs.append(hist_h2h_df['away_team_avg_ADR'][0])
                    adr_stds.append(hist_h2h_df['away_team_std_ADR'][0])
                    swings.append(hist_h2h_df['away_team_avg_Swing'][0])
                    swing_stds.append(hist_h2h_df['away_team_std_Swing'][0])
                    maps_total.append(hist_h2h_df['maps_played'][0])
                    maps_wins.append(hist_h2h_df['away_maps_won'][0])
                    maps_picked.append(hist_h2h_df['maps_played'][0] - hist_h2h_df['home_maps_picked'][0])
                    maps_avg_score_diffs.append( - hist_h2h_df['map_avg_score_diff'][0])
                    opp = hist_h2h_df['home_team'][0]
                else:
                    print("ERROR: team_id not in df")

                # HLTV Rank & points
                rank_data_opp = get_rank_data(match['date_parsed'], opp)

                ranks_opp.append(rank_data_opp['rank'])
                ranks_change.append(rank_data_opp['rank_change'])
                rank_diffs.append(rank_data_opp['rank'] - rank_data_side['rank'])
                points_opp.append(rank_data_opp['points'])
                points_change.append(rank_data_opp['points_change'])
                point_diffs.append(rank_data_side['points'] - rank_data_opp['points'])
                            
            except Exception as e:
                print(f"    ⚠️ H2H scraping hiba: {e}")
                continue
        
        # Átlagok számítása
        avg_h2h_winrate = np.sum(h2h_wins) / np.sum(h2h_total) if h2h_total else None
        avg_rating = np.mean(ratings) if ratings else None
        avg_rating_std = np.mean(rating_stds) if rating_stds else None
        avg_adr = np.mean(adrs) if adrs else None
        avg_adr_std = np.mean(adr_stds) if adr_stds else None
        avg_swing = np.mean(swings) if swings else None
        avg_swing_std = np.mean(swing_stds) if swing_stds else None
        avg_maps_total = np.mean(maps_total) if maps_total else None
        avg_map_winrate = np.sum(maps_wins) / np.sum(maps_total) if maps_wins else None
        avg_map_pickrate = np.sum(maps_picked) / np.sum(maps_total) if maps_picked else None
        avg_map_score_diff = np.mean(maps_avg_score_diffs) if maps_avg_score_diffs else None
        avg_rank_diff = np.mean(rank_diffs) if rank_diffs else None
        std_rank_diff = np.std(rank_diffs) if rank_diffs else None
        avg_point_diff = np.mean(point_diffs) if point_diffs else None
        std_point_diff = np.std(point_diffs) if point_diffs else None
        weak_opp_rank_diff = np.max(rank_diffs) if rank_diffs else None
        strong_opp_rank_diff = np.min(rank_diffs) if rank_diffs else None
        weak_opp_point_diff = np.max(point_diffs) if point_diffs else None
        strong_opp_point_diff = np.min(point_diffs) if point_diffs else None

        historical_h2h[side] = {
            'avg_h2h_winrate': avg_h2h_winrate,
            'avg_rating': avg_rating,
            'avg_rating_std': avg_rating_std,
            'avg_adr': avg_adr,
            'avg_adr_std': avg_adr_std,
            'avg_swing': avg_swing,
            'avg_swing_std': avg_swing_std,
            'avg_maps_total': avg_maps_total,
            'avg_map_winrate': avg_map_winrate,
            'avg_map_pickrate': avg_map_pickrate,
            'avg_map_score_diff': avg_map_score_diff,
            'avg_rank_diff': avg_rank_diff,
            'std_rank_diff': std_rank_diff,
            'avg_point_diff': avg_point_diff,
            'std_point_diff': std_point_diff,
            'weak_opp_rank_diff': weak_opp_rank_diff,
            'strong_opp_rank_diff': strong_opp_rank_diff,
            'weak_opp_point_diff': weak_opp_point_diff,
            'strong_opp_point_diff': strong_opp_point_diff,
            'n_matches_scraped': len(ratings)
        }

    print(f"\n📊 Historical H2H stats összegzés:")
    for side in ['home', 'away']:
        stats = historical_h2h[side]
        display(stats)

    return historical_h2h, rank_data

Rank data function initialized.


In [7]:
# 10. ROLLING FEATURES SZÁMÍTÁSA

def _calculate_rolling_features(teams, team_histories, historical_h2h):

    print("\n" + "="*60)
    print("8️⃣ ROLLING FEATURES SZÁMÍTÁSA")
    print("="*60)

    ml_input_row = {}

    for side in ['home', 'away']:
        team_name = teams[side]['team_name']
        print(f"\n📊 {team_name} rolling features:")
        
        history = team_histories.get(side, pd.DataFrame())
        
        if history.empty:
            print(f"  ⚠️ Nincs history adat")
            continue
        
        # Last 3 winrate
        last_3 = history.head(3)
        last_3_wr = (last_3['result'] == 'win').mean() if len(last_3) > 0 else None
        
        # Last 5 winrate  
        last_5 = history.head(5)
        last_5_wr = (last_5['result'] == 'win').mean() if len(last_5) > 0 else None
        
        # Avg scores
        last_3_avg_for = last_3['score_for'].mean() if len(last_3) > 0 else None
        last_3_avg_against = last_3['score_against'].mean() if len(last_3) > 0 else None
        
        # Current streak
        streak = 0
        if len(history) > 0:
            last_result = history.iloc[0]['result']
            for _, match in history.iterrows():
                if match['result'] == last_result:
                    streak += 1
                else:
                    break
            if last_result == 'loss':
                streak *= -1
        
        print(f"  Last 3 winrate:    {last_3_wr:.3f}" if last_3_wr else "  Last 3 winrate:    N/A")
        print(f"  Last 5 winrate:    {last_5_wr:.3f}" if last_5_wr else "  Last 5 winrate:    N/A")
        print(f"  Last 3 avg score:  {last_3_avg_for:.1f} - {last_3_avg_against:.1f}" if last_3_avg_for else "  Last 3 avg score:  N/A")
        print(f"  Current streak:    {streak:+d}")
        
        # Store
        ml_input_row[f'{side}_last_3_winrate'] = last_3_wr
        ml_input_row[f'{side}_last_5_winrate'] = last_5_wr
        ml_input_row[f'{side}_last_3_avg_score_for'] = last_3_avg_for
        ml_input_row[f'{side}_last_3_avg_score_against'] = last_3_avg_against
        ml_input_row[f'{side}_current_streak'] = streak

        for key in historical_h2h[side].keys():
            value = historical_h2h[side][key]
            ml_input_row[f'{side}_{key}'] = value
            print(f"  {side}_{key}: {value:.2f}")

    # Difference features
    if 'home_last_3_winrate' in ml_input_row and 'away_last_3_winrate' in ml_input_row:
        diff_last3_wr = ml_input_row['home_last_3_winrate'] - ml_input_row['away_last_3_winrate']
        ml_input_row['diff_last_3_winrate'] = diff_last3_wr
        print(f"\n📊 Difference features:")
        print(f"  Diff last 3 WR:    {diff_last3_wr:+.3f}")

    return ml_input_row

In [8]:
# 11. RANKINGS ÉS EGYÉB FEATURE-ÖK

def _add_other_features(ml_input_row, event_id, selected_match, teams, rank_data, match_h2h):

    print("\n" + "="*60)
    print("9️⃣ RANKINGS ÉS EGYÉB FEATURE-ÖK")
    print("="*60)

    # Basic match info
    ml_input_row['match_id'] = selected_match['match_id']
    ml_input_row['event_id'] = event_id
    ml_input_row['date'] = selected_match['date_parsed']
    ml_input_row['match_url'] = selected_match['link']


    # Teams
    ml_input_row['team_home'] = teams['home']['team_name']
    ml_input_row['team_away'] = teams['away']['team_name']

    # Rankings
    ml_input_row['home_current_rank'] = rank_data['home']['rank']
    ml_input_row['away_current_rank'] = rank_data['away']['rank'] 
    ml_input_row['home_rank_change'] = rank_data['home']['rank_change']
    ml_input_row['away_rank_change'] = rank_data['away']['rank_change']

    print(f"  Home rank: #{ml_input_row['home_current_rank']} (change: {ml_input_row['home_rank_change']:+d})")
    print(f"  Away rank: #{ml_input_row['away_current_rank']} (change: {ml_input_row['away_rank_change']:+d})")

    # Date features
    ml_input_row['date_month'] = selected_match['date_parsed'].strftime("%m")
    ml_input_row['date_day'] = int(selected_match['date_parsed'].strftime("%w")) + 1

    # H2H features
    ml_input_row['H2H_winrate_team1'] = match_h2h['home_win_rate']
    ml_input_row['H2H_games'] = match_h2h['wins_home'] + match_h2h['wins_away']

    # Match info
    ml_input_row['match_rounds'] = selected_match['rounds']

    # Label (score)
    ml_input_row['score_home'] = selected_match['score_home']
    ml_input_row['score_away'] = selected_match['score_away']
    ml_input_row['label_home_win'] = 1 if selected_match['score_home'] > selected_match['score_away'] else 0

    print("✅ Feature-ök összegyűjtve!")

    return ml_input_row

In [9]:
# Find match and parse odds

def _parse_odds(ml_input_row):
    df_odds = pd.read_csv("odds.csv")

    df_odds.Date = pd.to_datetime(df_odds.Date)

    date = ml_input_row['date']
    oneday = timedelta(days=1.5)
    home_hltv = ml_input_row['team_home']
    away_hltv = ml_input_row['team_away']

    odds_row = df_odds[
                    (df_odds.event_id == int(ml_input_row['event_id'])) &
                    (df_odds.Date <= date + oneday) & 
                    (df_odds.Date >= date - oneday) &
                            
                    (
                        ((df_odds.home_team_mapped == home_hltv) & 
                        (df_odds.away_team_mapped == away_hltv) ) |
                            
                        ((df_odds.home_team_mapped == away_hltv) & 
                        (df_odds.away_team_mapped == home_hltv) ) 
                    )
                    ]
    
    display(odds_row)

    if len(odds_row) != 1:
        print(f"Length is not 1! ({len(odds_row)})")
    else:
        op_home = odds_row['home_team_mapped'].iloc[0]
        op_away = odds_row['away_team_mapped'].iloc[0]
        op_home_odds = odds_row['home_odds'].iloc[0]
        op_away_odds = odds_row['away_odds'].iloc[0]

        if (home_hltv == op_home) & (away_hltv == op_away):
            ml_input_row['home_odds'] = op_home_odds
            ml_input_row['away_odds'] = op_away_odds
            ml_input_row['home_implied_odds'] = 1 / op_home_odds
            ml_input_row['away_implied_odds'] = 1 / op_away_odds
        elif (away_hltv == op_home) & (home_hltv == op_away):
            ml_input_row['home_odds'] = op_away_odds
            ml_input_row['away_odds'] = op_home_odds
            ml_input_row['home_implied_odds'] = 1 / op_away_odds
            ml_input_row['away_implied_odds'] = 1 / op_home_odds
        else:
            print("ERROR")

    return ml_input_row

In [10]:
# CSV manager

def match_id_duplicate(selected_match, save_path = "data/ml_input.csv"):
    # Load
    df = pd.read_csv(save_path)
    display(df)
    if df.empty:
        return False

    # Check id
    if int(selected_match['match_id']) in df['match_id'].values:
        print(f"Match ID ({selected_match['match_id']}) already exists!")
        return True
    else:
        return False

def update_csv(ml_input_row, save_path = "data/ml_input.csv"):
    # Load
    df = pd.read_csv(save_path)

    # Append
    new_row = pd.DataFrame.from_dict(ml_input_row, orient='index').T
    df = pd.concat([df, new_row], ignore_index=True)
    # Check duplicates
    df = df.drop_duplicates(subset=['match_id', 'team_home', 'team_away'],
                            keep='first')

    # Save
    df.to_csv(save_path, index=False)
    print(f"✅ Új sor mentve. ({len(df)})")

In [14]:
# MAIN

for event_id in tours.keys():
    target_matches = _target_match_scrape(event_id)

    for _, selected_match in target_matches.iterrows():
        
        if match_id_duplicate(selected_match):
            continue
        
        print(f"🎯 Kiválasztott meccs (index={_}):")
        print(f"  Match ID:   {selected_match['match_id']}")
        print(f"  Date:       {selected_match['date_parsed']}")
        print(f"  Teams:      {selected_match['team_home']} vs {selected_match['team_away']}")
        print(f"  Score:      {selected_match['score_home']} - {selected_match['score_away']}")
        print(f"  URL:        {selected_match['link']}")
        print()

        match_h2h = _h2h_match_scrape(selected_match)
        teams = _extract_teams(match_h2h)
        team_histories = _team_history_scrape(teams, selected_match)
        historical_h2h, rank_data = _historical_h2h_scrape(teams, team_histories, selected_match)
        ml_input_row1 = _calculate_rolling_features(teams, team_histories, historical_h2h)
        ml_input_row = _add_other_features(ml_input_row1, event_id, selected_match, teams, rank_data, match_h2h)
        ml_input_row = _parse_odds(ml_input_row)

        # DataFrame-ként megjelenítés
        ml_df = pd.DataFrame([ml_input_row])

        print("📊 DataFrame nézet:")
        display(ml_df.T.style.set_caption("ML Input Row - Transposed"))

        update_csv(ml_input_row, save_path = "data/ml_input.csv")


1️⃣ TARGET MATCHES SCRAPING
✅ 40 meccs találva


,match_id,event_id,date,team_home,team_away,score_home,score_away,map_type,rounds,link,date_clean,date_parsed
0,2380078,8292,March 16th 2025,MOUZ,Vitality,0,3,bo5,5,https://www.hltv.org/matches/2380078/mouz-vs-v...,March 16 2025,2025-03-16
1,2380076,8292,March 15th 2025,Vitality,The MongolZ,2,1,bo3,3,https://www.hltv.org/matches/2380076/vitality-...,March 15 2025,2025-03-15
2,2380077,8292,March 15th 2025,Spirit,MOUZ,1,2,bo3,3,https://www.hltv.org/matches/2380077/spirit-vs...,March 15 2025,2025-03-15


,match_id,event_id,date,team_home,team_away,match_url,home_last_3_winrate,home_last_5_winrate,home_last_3_avg_score_for,home_last_3_avg_score_against,...,H2H_winrate_team1,H2H_games,match_rounds,score_home,score_away,label_home_win,home_odds,away_odds,home_implied_odds,away_implied_odds
0,2380078,8292,2025-03-16 00:00:00,MOUZ,Vitality,https://www.hltv.org/matches/2380078/mouz-vs-v...,1.000000,0.8,2.000000,0.666667,...,0.3571,14,5,0,3,0,3.70,1.26,0.270270,0.793651
1,2380076,8292,2025-03-15 00:00:00,Vitality,The MongolZ,https://www.hltv.org/matches/2380076/vitality-...,1.000000,1.0,2.000000,0.000000,...,1.0000,7,3,2,1,1,1.27,3.66,0.787402,0.273224
2,2380077,8292,2025-03-15 00:00:00,Spirit,MOUZ,https://www.hltv.org/matches/2380077/spirit-vs...,1.000000,0.8,2.000000,0.333333,...,0.5333,15,3,1,2,0,1.37,2.96,0.729927,0.337838
3,2380073,8292,2025-03-14 00:00:00,Natus Vincere,The MongolZ,https://www.hltv.org/matches/2380073/natus-vin...,NaN,NaN,NaN,NaN,...,1.0000,3,3,0,2,0,1.47,2.65,0.680272,0.377358
4,2380072,8292,2025-03-14 00:00:00,Vitality,Liquid,https://www.hltv.org/matches/2380072/vitality-...,NaN,NaN,NaN,NaN,...,0.7500,8,3,2,0,1,1.20,4.35,0.833333,0.229885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,2381764,8036,2025-04-23 00:00:00,Liquid,MIBR,https://www.hltv.org/matches/2381764/liquid-vs...,0.333333,0.2,0.666667,1.666667,...,0.6667,3,3,2,0,1,1.44,2.69,0.694444,0.371747
102,2381762,8036,2025-04-23 00:00:00,Complexity,GamerLegion,https://www.hltv.org/matches/2381762/complexit...,0.666667,0.6,1.666667,0.666667,...,NaN,0,3,1,2,0,1.77,2.00,0.564972,0.500000
103,2381761,8036,2025-04-23 00:00:00,FaZe,3DMAX,https://www.hltv.org/matches/2381761/faze-vs-3...,0.666667,0.6,1.333333,0.666667,...,1.0000,3,3,2,0,1,1.52,2.47,0.657895,0.404858
104,2381758,8036,2025-04-22 00:00:00,SAW,MIBR,https://www.hltv.org/matches/2381758/saw-vs-mi...,0.333333,0.4,0.666667,1.666667,...,NaN,0,3,1,2,0,2.09,1.70,0.478469,0.588235


Match ID (2380078) already exists!


,match_id,event_id,date,team_home,team_away,match_url,home_last_3_winrate,home_last_5_winrate,home_last_3_avg_score_for,home_last_3_avg_score_against,...,H2H_winrate_team1,H2H_games,match_rounds,score_home,score_away,label_home_win,home_odds,away_odds,home_implied_odds,away_implied_odds
0,2380078,8292,2025-03-16 00:00:00,MOUZ,Vitality,https://www.hltv.org/matches/2380078/mouz-vs-v...,1.000000,0.8,2.000000,0.666667,...,0.3571,14,5,0,3,0,3.70,1.26,0.270270,0.793651
1,2380076,8292,2025-03-15 00:00:00,Vitality,The MongolZ,https://www.hltv.org/matches/2380076/vitality-...,1.000000,1.0,2.000000,0.000000,...,1.0000,7,3,2,1,1,1.27,3.66,0.787402,0.273224
2,2380077,8292,2025-03-15 00:00:00,Spirit,MOUZ,https://www.hltv.org/matches/2380077/spirit-vs...,1.000000,0.8,2.000000,0.333333,...,0.5333,15,3,1,2,0,1.37,2.96,0.729927,0.337838
3,2380073,8292,2025-03-14 00:00:00,Natus Vincere,The MongolZ,https://www.hltv.org/matches/2380073/natus-vin...,NaN,NaN,NaN,NaN,...,1.0000,3,3,0,2,0,1.47,2.65,0.680272,0.377358
4,2380072,8292,2025-03-14 00:00:00,Vitality,Liquid,https://www.hltv.org/matches/2380072/vitality-...,NaN,NaN,NaN,NaN,...,0.7500,8,3,2,0,1,1.20,4.35,0.833333,0.229885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,2381764,8036,2025-04-23 00:00:00,Liquid,MIBR,https://www.hltv.org/matches/2381764/liquid-vs...,0.333333,0.2,0.666667,1.666667,...,0.6667,3,3,2,0,1,1.44,2.69,0.694444,0.371747
102,2381762,8036,2025-04-23 00:00:00,Complexity,GamerLegion,https://www.hltv.org/matches/2381762/complexit...,0.666667,0.6,1.666667,0.666667,...,NaN,0,3,1,2,0,1.77,2.00,0.564972,0.500000
103,2381761,8036,2025-04-23 00:00:00,FaZe,3DMAX,https://www.hltv.org/matches/2381761/faze-vs-3...,0.666667,0.6,1.333333,0.666667,...,1.0000,3,3,2,0,1,1.52,2.47,0.657895,0.404858
104,2381758,8036,2025-04-22 00:00:00,SAW,MIBR,https://www.hltv.org/matches/2381758/saw-vs-mi...,0.333333,0.4,0.666667,1.666667,...,NaN,0,3,1,2,0,2.09,1.70,0.478469,0.588235


...


8️⃣ ROLLING FEATURES SZÁMÍTÁSA

📊 FaZe rolling features:
  Last 3 winrate:    0.667
  Last 5 winrate:    0.600
  Last 3 avg score:  1.7 - 0.7
  Current streak:    +1
  home_avg_h2h_winrate: 0.47
  home_avg_rating: 1.17
  home_avg_rating_std: 0.25
  home_avg_adr: 77.97
  home_avg_adr_std: 13.93
  home_avg_swing: 1.80
  home_avg_swing_std: 3.07
  home_avg_maps_total: 2.33
  home_avg_map_winrate: 0.71
  home_avg_map_pickrate: 0.57
  home_avg_map_score_diff: 3.22
  home_avg_rank_diff: 6.00
  home_std_rank_diff: 5.10
  home_avg_point_diff: 166.67
  home_std_point_diff: 66.43
  home_weak_opp_rank_diff: 13.00
  home_strong_opp_rank_diff: 1.00
  home_weak_opp_point_diff: 254.00
  home_strong_opp_point_diff: 93.00
  home_n_matches_scraped: 3.00

📊 paiN rolling features:
  Last 3 winrate:    N/A
  Last 5 winrate:    N/A
  Last 3 avg score:  0.7 - 2.0
  Current streak:    -5
  away_avg_h2h_winrate: 0.44
  away_avg_rating: 0.95
  away_avg_rating_std: 0.20
  away_avg_adr: 69.60
  away_avg_adr_std:

,Date,home_team,away_team,home_odds,away_odds,event_id,home_team_mapped,away_team_mapped
204,2025-04-21,FaZe Clan,paiN Gaming,1.04,9.06,8036,FaZe,paiN


📊 DataFrame nézet:


,0
home_last_3_winrate,0.666667
home_last_5_winrate,0.600000
home_last_3_avg_score_for,1.666667
home_last_3_avg_score_against,0.666667
home_current_streak,1
home_avg_h2h_winrate,0.473684
home_avg_rating,1.172000
home_avg_rating_std,0.247067
home_avg_adr,77.966667
home_avg_adr_std,13.926667


✅ Új sor mentve. (120)


# Other

## Rankings history

In [ ]:
# Rankings history

# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings(url):
    driver = webdriver.Chrome()
    driver.get(url)
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    url_prev = driver.find_element(By.CLASS_NAME, "pagination-prev ").get_attribute("href")

    date_raw = driver.find_element(By.CLASS_NAME, "regional-ranking-header-text").text
    date = date_raw.split("ranking on ")[-1]
    # ordinal ragok eltávolítása
    date_clean = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date)
    # dátummá alakítás
    date_parsed = pd.to_datetime(date_clean, format="%B %d, %Y")

    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "").replace(" points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': date_parsed,
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings), url_prev

# Összes rankings leszedése
all_rankings = pd.DataFrame()
url = "https://www.hltv.org/ranking/teams/"
url_prev = ""
while url_prev != "https://www.hltv.org/ranking/teams/2024/december/30":
    rankings, url_prev = scrape_team_rankings(url)
    print(f"Scraping: {url}")
    all_rankings = pd.concat([all_rankings, rankings], ignore_index=True)
    display(rankings.sample(3))
    print(f"+{len(rankings)} (Total: {len(all_rankings)})")
    url = url_prev

In [ ]:
# Save rankings history

all_rankings.to_csv("data/all_rankings2025.csv", index=False)

## OddsPortal

In [ ]:
# Tournaments

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_oddsportal_cs_links():
    url = "https://www.oddsportal.com/results/#esports"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    # várjuk, amíg betöltődik legalább 1 tournament link
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")))

    time.sleep(2)  # kis extra wait, hogy minden JS lefusson

    links = driver.find_elements(By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")

    data = []
    for link in links:
        try:
            text = link.text.strip()
            href = link.get_attribute("href")
            if text.startswith("Counter-Strike "):
                name = text.replace("Counter-Strike ", "").strip()
                data.append({
                    "Name": name,
                    "url": href
                })
        except Exception as e:
            print(f"⚠️ Hiba egy link feldolgozásánál: {e}")
            continue

    driver.quit()
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} Counter-Strike esemény található az OddsPortalon.")
    return df


# --- Példa futtatás ---
df_oddsportal = scrape_oddsportal_cs_links()
display(df_oddsportal.sample(5))


In [ ]:
# Search tournaments

#df_oddsportal[df_oddsportal.Name == "ESL Pro League Season 21"]["url"].iloc[0]
#df_oddsportal[df_oddsportal.Name.str.contains("IEM")].loc[196, "url"]

In [ ]:
# Odds of tournaments

tours = {'8292': {'name': 'ESL Pro League Season 21',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-esl-pro-league-season-21/results/'},
         '7907': {'name': 'BLAST Open London 2025',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-blast-open/results/'},
         '8038': {'name': 'IEM Cologne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-intel-extreme-masters-cologne/results/'},
         '8036': {'name': 'IEM Melbourne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-iem-melbourne/results/'}
        }


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re, time
from datetime import datetime

def scrape_oddsportal_fixed(event_url, headless=False):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0")
    
    driver = webdriver.Chrome(options=opts)
    driver.get(event_url)
    wait = WebDriverWait(driver, 20)
    
    all_data = []
    page = 1
    
    print(f"Scrape: {event_url}")
    while True:
        print(f"🔍 Oldal {page} feldolgozása...")
        
        # Fokozatos scroll (több elemet tölt be)
        print(f"  📜 Scrolling...")
        for i in range(2):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 4);")
            time.sleep(1.5)  # Lassabb scroll

        # Extra várakozás a renderelésre
        time.sleep(3)
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.eventRow")))

        # Extra wait hogy a game-row-k is betöltsenek
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testid='game-row']")))
        except:
            print("  ⚠️ Nincs game-row az oldalon")

        event_rows = driver.find_elements(By.CSS_SELECTOR, "div.eventRow")
        all_game_rows_global = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='game-row']")

        # Csak az eventRow-n BELÜLI game-row-kat számoljuk
        game_rows_in_events = []
        for event in event_rows:
            try:
                game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
                game_rows_in_events.append(game_row)
            except:
                pass

        print(f"  📊 {len(event_rows)} eventRow")
        print(f"  📊 {len(all_game_rows_global)} game-row (GLOBAL - rossz!)")
        print(f"  📊 {len(game_rows_in_events)} game-row (eventRow-kban - jó!)")

        current_date = None
        matches_processed_this_page = 0

        # Dátum keresése MINDEN eventRow-ban - JAVÍTOTT VERZIÓ
        for event in event_rows:
            # dátum keresése data-testid="date-header" elemben
            try:
                date_header = event.find_element(By.CSS_SELECTOR, '[data-testid="date-header"]')
                date_text = date_header.text.strip()
                if any(month in date_text for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                    # Dátum formázása - eltávolítjuk a " -" utáni részt
                    clean_date = date_text.split(' -')[0].strip()
                    current_date = clean_date
                    
                    # Dátum átalakítása YYYY-MM-dd formátumba
                    try:
                        date_obj = datetime.strptime(current_date, '%d %b %Y')
                        current_date = date_obj.strftime('%Y-%m-%d')
                    except ValueError:
                        # Ha nem sikerül átalakítani, marad az eredeti
                        pass
                    # NE continue-oljunk itt! Hagyjuk, hogy a game-row keresés is lefusson
            except:
                pass  # Nincs date-header ebben az eventRow-ban

            # Ha nincs dátum, de van game-row, akkor meccs sor
            try:
                game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
                #print(game_row.text)
            except:
                continue

            # DEBUG: Van game-row, nézzük mi van benne
            # csapatnevek
            participants = game_row.find_elements(By.CSS_SELECTOR, "a[title]")
            if len(participants) < 2:
                print(f"    ⚠️ SKIP: Kevés participant ({len(participants)}) - nincs 2 csapat")
                continue
            
            home_team = participants[0].get_attribute("title").strip()
            away_team = participants[1].get_attribute("title").strip()
            
            if not home_team or not away_team:
                print(f"    ⚠️ SKIP: Üres csapatnév (home={home_team}, away={away_team})")
                continue
            
            # oddsok
            odds_blocks = event.find_elements(By.CSS_SELECTOR, "div[data-testid^='odd-container'] p")
            odds = []
            for p in odds_blocks:
                try:
                    odds_text = p.text.strip().replace(",", ".")
                    if re.match(r"^\d+(\.\d+)?$", odds_text):
                        odds.append(float(odds_text))
                except:
                    continue
            
            home_odds = odds[0] if len(odds) > 0 else None
            away_odds = odds[1] if len(odds) > 1 else None
            
            if not home_odds or not away_odds:
                print(f"    ⚠️ SKIP: Nincs odds (home={home_odds}, away={away_odds}) - {home_team} vs {away_team}")
                continue
            
            if not current_date:
                print(f"    ⚠️ WARNING: Nincs dátum! - {home_team} vs {away_team}")
                current_date = "Unknown date"  # ← Csak ezt a sort kell hozzáadni

            all_data.append({
                "Date": current_date,
                "home_team": home_team,
                "away_team": away_team,
                "home_odds": home_odds,
                "away_odds": away_odds
            })

            matches_processed_this_page += 1  # ← Számláló növelése

        # Következő oldal ellenőrzése
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, f"a.pagination-link[data-number='{page + 1}']")
            if next_button.is_enabled():
                print(f"➡️ Következő oldal: {page + 1}")
                driver.execute_script("arguments[0].click();", next_button)
                page += 1
                time.sleep(3)  # Várakozás az oldal betöltésére
                continue
            else:
                break
        except:
            # Ha nincs következő oldal, kilépünk
            break

    driver.quit()
    
    df = pd.DataFrame(all_data).drop_duplicates(subset=["Date","home_team","away_team","home_odds","away_odds"])
    print(f"✅ Összesen {len(df)} meccs feldolgozva {page} oldalról.")
    return df

# Futtatás
df_odds = pd.DataFrame()
for id in tours.keys():
    print(f"Scraping {tours[id]['name']}")
    url_tour = tours[id]['url']
    df_odds_tour = scrape_oddsportal_fixed(url_tour, headless=False)
    df_odds_tour.Date = pd.to_datetime(df_odds_tour.Date)
    df_odds_tour['event_id'] = id
    display(df_odds_tour.sample(5))
    
    df_odds = pd.concat([df_odds, df_odds_tour], ignore_index=True)

print(f"Total odds found {len(df_odds)}")


In [ ]:
# Fuzzy matching

import pandas as pd
from fuzzywuzzy import process, fuzz
import json
from pathlib import Path

def fuzzy_match_teams(odds_teams, ranking_teams, save_path="fuzzy_mapping.json", threshold=70):
    """
    Fuzzy matching odds csapatok → rankings csapatok
    
    Args:
        odds_teams: list of team names from odds data
        ranking_teams: list of team names from rankings data  
        save_path: path to save the mapping
        threshold: minimum similarity score (0-100)
    """
    
    # Betöltés korábbi mapping-ből, ha létezik
    mapping = {}
    if Path(save_path).exists():
        with open(save_path, 'r', encoding='utf-8') as f:
            mapping = json.load(f)
        print(f"✅ Korábbi mapping betöltve: {len(mapping)} csapat")
    
    # Csak azokat a csapatokat dolgozzuk fel, amik még nincsenek a mapping-ben
    teams_to_process = [team for team in odds_teams if team not in mapping]
    
    if not teams_to_process:
        print("✅ Minden csapat már mapped!")
        return mapping
    
    print(f"🔍 {len(teams_to_process)} csapat feldolgozása...")
    
    for i, team in enumerate(teams_to_process, 1):
        print(f"\n[{i}/{len(teams_to_process)}] {team}")
        
        # Pontos egyezés
        if team in ranking_teams:
            mapping[team] = team
            print(f"   ✅ Auto match: {team}")
            continue
        
        # Fuzzy matching
        matches = process.extract(team, ranking_teams, limit=5, scorer=fuzz.token_sort_ratio)
        
        # Szűrés threshold alapján
        good_matches = [(match, score) for match, score in matches if score >= threshold]
        
        if len(good_matches) == 1:
            # Egyértelmű match
            best_match, best_score = good_matches[0]
            mapping[team] = best_match
            print(f"   ✅ Auto fuzzy: {team} → {best_match} ({best_score})")
        
        elif len(good_matches) > 1:
            # Több jó match - user dönt
            best_match, best_score = good_matches[0]
            second_match, second_score = good_matches[1]
            
            # Egyértelmű, ha nagy a különbség
            if best_score - second_score >= 15:
                mapping[team] = best_match
                print(f"   ✅ Clear winner: {team} → {best_match} ({best_score})")
            else:
                # User választ
                print(f"   ❓ Több hasonló találat:")
                for j, (match, score) in enumerate(good_matches, 1):
                    print(f"      {j}. {match} ({score})")
                print(f"      0. SKIP (kézi feldolgozás később)")
                
                try:
                    choice = input(f"   Válassz [1-{len(good_matches)}]: ").strip()
                    if choice.isdigit():
                        choice_int = int(choice)
                        if 1 <= choice_int <= len(good_matches):
                            chosen_match = good_matches[choice_int - 1][0]
                            mapping[team] = chosen_match
                            print(f"   👉 Kiválasztva: {team} → {chosen_match}")
                        else:
                            mapping[team] = None
                            print(f"   ⏭️ Skipped: {team}")
                    else:
                        mapping[team] = None
                        print(f"   ⏭️ Skipped: {team}")
                except:
                    mapping[team] = None
                    print(f"   ⏭️ Skipped: {team}")
        else:
            # Nincs jó match
            mapping[team] = None
            print(f"   ❌ No good match found for: {team}")
    
    # Mentés
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Mapping mentve: {save_path}")
    print(f"📊 Statisztika:")
    print(f"   Összes csapat: {len(mapping)}")
    print(f"   Sikeres match: {sum(1 for v in mapping.values() if v is not None)}")
    print(f"   Nincs match: {sum(1 for v in mapping.values() if v is None)}")
    
    return mapping

# Használat:
def apply_fuzzy_mapping(df_odds, rankings, mapping_path="fuzzy_mapping.json"):
    """Apply fuzzy mapping to merge odds with rankings"""
    
    # Betöltés mapping
    with open(mapping_path, 'r', encoding='utf-8') as f:
        mapping = json.load(f)
    
    # Csapatok összekapcsolása
    df_odds['home_team_mapped'] = df_odds['home_team'].map(mapping)
    df_odds['away_team_mapped'] = df_odds['away_team'].map(mapping)
    
    # Rankings merge
    rankings_clean = rankings.drop_duplicates('team_name')
    
    df_merged = df_odds.merge(
        rankings_clean, 
        left_on='home_team_mapped', 
        right_on='team_name', 
        how='left',
        suffixes=('', '_home')
    ).merge(
        rankings_clean,
        left_on='away_team_mapped', 
        right_on='team_name', 
        how='left',
        suffixes=('_home', '_away')
    )
    
    # Statisztika
    matched_home = df_merged['home_team_mapped'].notna().sum()
    matched_away = df_merged['away_team_mapped'].notna().sum()
    total_matches = len(df_merged)
    
    print(f"📊 Merge statisztika:")
    print(f"   Home team matched: {matched_home}/{total_matches} ({matched_home/total_matches*100:.1f}%)")
    print(f"   Away team matched: {matched_away}/{total_matches} ({matched_away/total_matches*100:.1f}%)")
    
    return df_merged

# PÉLDA HASZNÁLAT:
if __name__ == "__main__":
    # 2. Csapatok listázása
    odds_teams = pd.unique([*df_odds.home_team.unique(), *df_odds.away_team.unique()])
    ranking_teams = rankings.team_name.unique()
    
    print(f"📊 Adatok:")
    print(f"   Odds csapatok: {len(odds_teams)}")
    print(f"   Ranking csapatok: {len(ranking_teams)}")
    
    # 3. Fuzzy matching futtatása
    mapping = fuzzy_match_teams(odds_teams, ranking_teams, "team_mapping.json")
    
    # 4. Merge elvégzése
    df_final = apply_fuzzy_mapping(df_odds, rankings, "team_mapping.json")

In [ ]:
# Save dem odds

df_odds.to_csv("odds.csv", index=False)

In [ ]:
# Manual search
for t in rankings.team_name.unique():
    if 'nrg' in t.lower():
        print(t)

In [ ]:
# Check if names in dataframes

with open("team_mapping.json", 'r', encoding='utf-8') as f:
    mapping = json.load(f)

for t_op, t_hltv in mapping.items():
    if (t_op in odds_teams) & (t_hltv in rankings.team_name.unique()):
        pass
    else:
        print(f"Error for {t_op}/{t_hltv}")